# LeetCode 3949 — Subtree Inversion Sum II bằng Python

- Link: https://leetcode.com/problems/subtree-inversion-sum-ii/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động trên cây (tree DP), khoảng cách trên cây, tối ưu phép gộp bằng max/min hậu tố

> **Bài Premium: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**
> Bài Premium không có skeleton công khai. Tên phương thức `subtreeInversionSum(self, edges, nums, k)` được suy luận (theo bài phiên bản I cùng họ và tên hàm thường gặp) — **cần đối chiếu** lại chữ ký trên LeetCode khi nộp.

Skeleton (suy luận):

```python
class Solution:
    def subtreeInversionSum(self, edges: List[List[int]], nums: List[int], k: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho một cây gốc `0`, mỗi nút có một giá trị (có thể âm). Ta được chọn một tập nút để **đảo dấu cả cây con** của chúng (nhân `−1` mọi giá trị trong cây con). Điều kiện: **hai nút bất kỳ** được chọn phải cách nhau **ít nhất `k` cạnh**. Cần tổng giá trị lớn nhất sau khi đảo.

Điểm khó: khoảng cách bị ràng buộc cho **mọi cặp** (cả hai nút anh em, họ hàng xa), không chỉ cặp tổ tiên – hậu duệ. Và phép đảo lồng nhau **triệt tiêu** nhau (đảo hai lần = không đảo).

Sau notebook này bạn sẽ:

1. Mô hình hóa dấu của mỗi nút là `(−1)^{số nút được chọn trên đường từ nó lên gốc, tính cả chính nó}`.
2. Thiết kế trạng thái DP `m` = khoảng cách từ nút hiện tại đến nút được chọn **gần nhất trong cây con** (chặn trên bởi `k`), và hiểu vì sao chỉ cần cặp gần nhất.
3. Theo dõi cả **max và min** của tổng, vì phép đảo của tổ tiên biến max thành min.
4. Tối ưu phép gộp con vào cha từ `O(k²)` xuống `O(k)` bằng max/min **hậu tố**, cho tổng `O(n·k)`.
5. Viết 3 lời giải (vét cạn tập con, DP gộp `O(k²)`, DP gộp `O(k)`) và kiểm chứng bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho cây vô hướng `n` nút (`0..n−1`) gốc `0`, mảng cạnh `edges`, mảng giá trị `nums`, số nguyên `k`.

- **Đảo một nút:** mọi giá trị trong cây con gốc tại nút đó nhân `−1`.
- Có thể đảo một tập nút tùy ý (kể cả rỗng), nhưng hai nút được đảo `a, b` phải có `dist(a, b) ≥ k` (số cạnh trên đường đi duy nhất).
- Trả về tổng lớn nhất của các giá trị sau khi đảo.

**Ví dụ 1:** `edges = [[0,1],[0,2],[0,3],[1,4],[1,5]]`, `nums = [1,0,-10,3,4,5]`, `k = 2` → `23` (đảo cây con của nút 2).

**Ví dụ 2:** `edges = [[0,1],[1,2]]`, `nums = [5,-10,-10]`, `k = 1` → `25` (đảo nút 1).

**Ví dụ 3:** `edges = [[0,1],[0,2]]`, `nums = [1,-5,-6]`, `k = 2` → `12` (đảo nút 1 và 2: cách nhau 2 cạnh ≥ k).

**Ví dụ 4:** `edges = [[0,1],[0,2]]`, `nums = [1,-5,-6]`, `k = 3` → `10` (đảo nút 0; không thể đảo 1 và 2 vì cách nhau 2 < 3).

**Ràng buộc:** `2 ≤ n ≤ 5·10^4`, `−4·10^4 ≤ nums[i] ≤ 4·10^4`, `1 ≤ k ≤ 50`, `edges` tạo thành cây.

In [1]:
import random, time
from itertools import combinations
from collections import deque

EXAMPLES = [
    (([[0,1],[0,2],[0,3],[1,4],[1,5]], [1,0,-10,3,4,5], 2), 23),
    (([[0,1],[1,2]], [5,-10,-10], 1), 25),
    (([[0,1],[0,2]], [1,-5,-6], 2), 12),
    (([[0,1],[0,2]], [1,-5,-6], 3), 10),
]
for (edges, nums, k), exp in EXAMPLES:
    print(f"n={len(nums)} k={k} -> kỳ vọng {exp}")

n=6 k=2 -> kỳ vọng 23
n=3 k=1 -> kỳ vọng 25
n=3 k=2 -> kỳ vọng 12
n=3 k=3 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. Dựng cây có gốc từ danh sách cạnh (BFS)

Đề cho cạnh **vô hướng**, nên ta cần xác định ai là cha ai khi gốc là `0`. Dùng danh sách kề + BFS: nút nào được thăm từ `u` thì có cha là `u`. Thứ tự BFS có tính chất: **cha luôn đứng trước con**. Duyệt thứ tự đó **ngược** ta xử lý mọi con trước cha — không cần đệ quy (tránh tràn stack với cây dạng đường thẳng 5·10^4 nút).

In [2]:
def rooted_tree(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    parent = [-1] * n
    order = [0]
    seen = [False] * n
    seen[0] = True
    for u in order:                      # order vừa duyệt vừa mở rộng: chính là BFS
        for w in adj[u]:
            if not seen[w]:
                seen[w] = True
                parent[w] = u
                order.append(w)
    return parent, order

parent, order = rooted_tree(6, [[0,1],[0,2],[0,3],[1,4],[1,5]])
print("parent =", parent)
print("order  =", order, "(cha luôn đứng trước con)")

parent = [-1, 0, 0, 0, 1, 1]
order  = [0, 1, 2, 3, 4, 5] (cha luôn đứng trước con)


### 3.2. Dấu của một nút sau các phép đảo

Nếu nút `u` được chọn, cả cây con của `u` bị nhân `−1`. Một nút `v` bị đảo một lần cho **mỗi nút được chọn là tổ tiên của nó hoặc chính nó**. Vậy:

```
giá trị cuối của v  =  nums[v] · (−1)^{ c(v) },   c(v) = số nút được chọn nằm trên đường v → gốc (gồm cả v)
```

Hai lần đảo triệt tiêu nhau. Ví dụ: đảo nút `1` và nút `4` (con của `1`): nút `4` bị đảo hai lần → giữ nguyên dấu.

In [3]:
def final_values(n, parent, nums, chosen):
    chosen = set(chosen)
    res = []
    for v in range(n):
        c, u = 0, v
        while u != -1:
            if u in chosen:
                c += 1
            u = parent[u]
        res.append(nums[v] * (-1) ** c)
    return res

edges, nums = [[0,1],[0,2],[0,3],[1,4],[1,5]], [1,0,-10,3,4,5]
parent, order = rooted_tree(6, edges)
print(final_values(6, parent, nums, [2]))        # đảo nút 2 -> -10 thành +10
print(final_values(6, parent, nums, [1, 4]))     # nút 4 bị đảo 2 lần -> giữ nguyên
print(final_values(6, parent, nums, [0, 1]))     # nút 1 bị đảo 2 lần (qua 0 và qua 1)

[1, 0, 10, 3, 4, 5]
[1, 0, -10, 3, 4, -5]
[-1, 0, 10, -3, 4, 5]


### 3.3. Khoảng cách giữa hai nút trên cây

`dist(a, b)` = số cạnh trên đường duy nhất. Tính bằng BFS từ `a`. Ta cần nó cho vét cạn kiểm tra điều kiện "mọi cặp nút được chọn cách nhau ≥ k".

In [4]:
def all_distances(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    dist = []
    for s in range(n):
        d = [-1] * n
        d[s] = 0
        q = deque([s])
        while q:
            u = q.popleft()
            for w in adj[u]:
                if d[w] < 0:
                    d[w] = d[u] + 1
                    q.append(w)
        dist.append(d)
    return dist

dist = all_distances(6, edges)
print("dist(2,3) =", dist[2][3], " dist(4,5) =", dist[4][5], " dist(0,4) =", dist[0][4])

dist(2,3) = 2  dist(4,5) = 2  dist(0,4) = 2


### 3.4. `float("-inf")` làm giá trị "không khả thi"

Trong DP, nhiều trạng thái không thể xảy ra (ví dụ "nút gần nhất được chọn cách 1" ở một lá). Ta đánh dấu bằng `-inf` (khi tìm max) hoặc `+inf` (khi tìm min). Cộng `-inf` với số hữu hạn vẫn là `-inf`, và `max(x, -inf) = x`, nên không cần `if` đặc biệt.

In [5]:
NEG, POS = float("-inf"), float("inf")
print(NEG + 5, max(7, NEG), POS + 5, min(7, POS))
print(max(NEG + 5, 3), -(POS))     # phép đảo dấu biến POS thành NEG

-inf 7 inf 7
3 -inf


### 3.5. Max hậu tố (suffix maximum)

`suf[i] = max(a[i], a[i+1], ..., a[cuối])`. Tính từ phải sang trái trong `O(len)`. Về sau mỗi truy vấn "max của mọi phần tử có chỉ số `≥ i`" chỉ còn `O(1)`.

In [6]:
a = [3, 1, 4, 1, 5, 2]
suf = [NEG] * (len(a) + 1)
for i in range(len(a) - 1, -1, -1):
    suf[i] = max(suf[i + 1], a[i])
print(suf)            # [5, 5, 5, 5, 5, 2, -inf]
print("max các phần tử từ chỉ số 2:", suf[2])

[5, 5, 5, 5, 5, 2, -inf]
max các phần tử từ chỉ số 2: 5


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 4**: nút `0` giá trị `1`, hai lá `1` và `2` giá trị `−5` và `−6`, `k = 3`.

Các tập chọn hợp lệ (mọi cặp cách nhau ≥ 3):

- Rỗng: tổng `1 − 5 − 6 = −10`.
- `{0}`: đảo cả cây: `−1 + 5 + 6 = 10`.
- `{1}`: `1 + 5 − 6 = 0`.
- `{2}`: `1 − 5 + 6 = 2`.
- `{1, 2}`: cách nhau 2 < 3 → **không hợp lệ**.
- `{0, 1}` hay `{0, 2}`: cách nhau 1 < 3 → không hợp lệ.

Đáp án `10`. Với `k = 2` thì `{1, 2}` hợp lệ (cách đúng 2): `1 + 5 + 6 = 12` (Ví dụ 3).

**Nhận xét quan trọng.** Với mọi nút `v`, điều ta cần biết về cây con của `v` để "nói chuyện" với phần còn lại của cây chỉ gồm:

1. **Khoảng cách từ `v` đến nút được chọn gần nhất trong cây con** (vì mọi ràng buộc từ bên ngoài đều đi qua `v`, nút gần nhất là nút nguy hiểm nhất).
2. **Tổng giá trị** của cây con (khi chưa tính đảo của tổ tiên). Vì tổ tiên đảo sẽ nhân tổng này với `−1`, ta cần biết cả **max và min** của tổng.

In [7]:
# Mọi tập chọn hợp lệ của ví dụ 4 (n = 3, k = 3), tính tổng bằng định nghĩa
edges, nums, k = [[0,1],[0,2]], [1,-5,-6], 3
parent, order = rooted_tree(3, edges)
dist = all_distances(3, edges)
for r in range(0, 4):
    for S in combinations(range(3), r):
        ok = all(dist[a][b] >= k for a, b in combinations(S, 2))
        total = sum(final_values(3, parent, nums, S))
        print(f"S={S!s:10} hợp lệ={ok!s:5} tổng={total}")

S=()         hợp lệ=True  tổng=-10
S=(0,)       hợp lệ=True  tổng=10
S=(1,)       hợp lệ=True  tổng=0
S=(2,)       hợp lệ=True  tổng=2
S=(0, 1)     hợp lệ=False tổng=0
S=(0, 2)     hợp lệ=False tổng=-2
S=(1, 2)     hợp lệ=False tổng=12
S=(0, 1, 2)  hợp lệ=False tổng=-12


## 5. Cách 1 — Brute Force: thử mọi tập con

**Ý tưởng.** `2^n` tập con các nút. Với mỗi tập: kiểm tra mọi cặp có `dist ≥ k` không; nếu có, tính tổng bằng định nghĩa ở mục 3.2; lấy `max`. Đây là bản dịch thẳng đề, dùng làm chuẩn đối chiếu.

In [8]:
def brute_force(edges, nums, k):
    n = len(nums)
    parent, order = rooted_tree(n, edges)
    dist = all_distances(n, edges)
    best = None
    for mask in range(1 << n):
        S = [i for i in range(n) if mask >> i & 1]
        if any(dist[a][b] < k for a, b in combinations(S, 2)):
            continue
        total = sum(final_values(n, parent, nums, S))
        if best is None or total > best:
            best = total
    return best

for (edges, nums, k), exp in EXAMPLES:
    got = brute_force(edges, nums, k)
    print(got, "(kỳ vọng", exp, ")")
    assert got == exp

23 (kỳ vọng 23 )
25 (kỳ vọng 25 )
12 (kỳ vọng 12 )
10 (kỳ vọng 10 )


**Độ phức tạp.** `O(2^n · (n² + n·depth))`. Chỉ dùng được cho `n ≤ 12`. **Nút thắt:** số tập con bùng nổ; cần một trạng thái gọn mô tả "chọn gì trong cây con" mà tập con đầy đủ không cần thiết.

## 6. Bước đột phá

### 6.1. Trạng thái DP

Với nút `v`, xét mọi cách chọn tập con trong **cây con của `v`** thỏa ràng buộc nội bộ. Với mỗi cách, định nghĩa:

- `m` = khoảng cách từ `v` đến nút được chọn **gần nhất** trong cây con của `v`; `m = 0` nếu chính `v` được chọn; và ta **chặn trên bằng `k`**: mọi giá trị `≥ k` (hoặc không có nút nào được chọn) gộp thành `m = k`.
- `val` = tổng giá trị cây con sau khi đảo, **chỉ tính các phép đảo bên trong cây con**.

Lưu `H[v][m] = max val` và `L[v][m] = min val` trên mọi cách chọn có khoảng cách `m`.

**Vì sao chặn `m` ở `k` được?** Mọi nút chọn ngoài cây con cách `v` ít nhất `1`, nên khoảng cách tới nút trong cây con ≥ `k + 1 > k`: luôn thỏa. Vậy phân biệt `m = k, k+1, ...` là thừa.

**Vì sao chỉ cần nút gần nhất?** Nếu cặp gần nhất đã cách ≥ k thì mọi cặp khác cũng vậy.

### 6.2. Gộp các con vào cha

Giữ `cur` = khoảng cách từ `v` đến nút chọn gần nhất trong các con **đã gộp**, kèm `hi[cur]`, `lo[cur]` = max/min của **tổng `val` của các con**. Mới đầu `cur = k` (chưa có gì), `hi = lo = 0`.

Gộp con `c` có `m_c`: khoảng cách từ `v` đến nút chọn gần nhất trong cây con `c` là `D = min(m_c + 1, k)`. **Hai nút chọn ở hai con khác nhau cách nhau `cur + D`** (đi qua `v`), nên cần:

```
cur + D ≥ k          (điều kiện hợp lệ)
cur' = min(cur, D)   (khoảng cách mới)
```

(Nếu `cur = k` hoặc `D = k` thì điều kiện luôn đúng vì cả hai ≥ 1.)

### 6.3. Quyết định cho chính `v`

Sau khi gộp hết các con ta có `hi[t], lo[t]` với `t ∈ [1, k]`:

- **Không chọn `v`:** `m_v = t`, `val = nums[v] + (tổng các con)` → `H[v][t] = nums[v] + hi[t]`, `L[v][t] = nums[v] + lo[t]`.
- **Chọn `v`:** hợp lệ khi mọi nút chọn trong các con cách `v` ít nhất `k`, tức `t = k`. Khi đó **toàn bộ cây con bị nhân `−1`**: `val = −(nums[v] + tổng các con)`, nên `H[v][0] = −(nums[v] + lo[k])` và `L[v][0] = −(nums[v] + hi[k])`. **Đây là chỗ cần min** để có max.

**Đáp án** = `max(H[0][m])` qua mọi `m` (gốc `0` không còn tổ tiên).

### 6.4. Tối ưu phép gộp

Gộp ngây thơ duyệt mọi cặp `(cur, D)` tốn `O(k²)` cho mỗi cạnh → `O(n·k²) = 5·10^4 · 2500 ≈ 1.25·10^8` cho mỗi mảng (max và min): quá chậm trong Python. Nhưng với `t` cố định, `cur' = t` xảy ra khi (A) `cur = t` và `D ≥ max(t, k−t)`, hoặc (B) `D = t` và `cur ≥ max(t, k−t)`. Mỗi trường hợp chỉ cần **max/min hậu tố** → `O(k)` cho mỗi cạnh. Đó là cách 3.

## 7. Các cách tiếp theo

### 7.1. Khung chung của hai cách DP

Cả hai cách DP dùng cùng một **khung**: duyệt các nút theo thứ tự BFS ngược, với mỗi nút `v`:

1. Lấy `hi, lo` đã tích lũy từ các con (hoặc khởi tạo rỗng).
2. Tính `H[v], L[v]` theo mục 6.3.
3. Đổi sang dạng "khoảng cách từ cha" `D` rồi **gộp ngay vào bộ tích lũy của cha** và vứt mảng của `v` đi (để tiết kiệm bộ nhớ).

Hai cách chỉ khác hàm `merge`.

In [9]:
def to_distance(Hc, Lc, k):
    # m_c in 0..k  ->  D = min(m_c + 1, k) in 1..k ; D = k gộp m_c = k-1 và m_c = k
    chH = [NEG] * (k + 1)
    chL = [POS] * (k + 1)
    for D in range(1, k):
        chH[D] = Hc[D - 1]
        chL[D] = Lc[D - 1]
    chH[k] = max(Hc[k - 1], Hc[k])
    chL[k] = min(Lc[k - 1], Lc[k])
    return chH, chL

def tree_dp(n, edges, nums, k, merge):
    parent, order = rooted_tree(n, edges)
    acc = [None] * n
    for v in reversed(order):
        if acc[v] is None:
            hi = [NEG] * (k + 1); lo = [POS] * (k + 1)
            hi[k] = lo[k] = 0                      # chưa có con nào: khoảng cách "k", tổng 0
        else:
            hi, lo = acc[v]
            acc[v] = None
        Hv = [NEG] * (k + 1); Lv = [POS] * (k + 1)
        for t in range(1, k + 1):                  # không chọn v
            Hv[t] = nums[v] + hi[t]
            Lv[t] = nums[v] + lo[t]
        Hv[0] = -(nums[v] + lo[k])                 # chọn v: cần mọi nút chọn ở con cách >= k
        Lv[0] = -(nums[v] + hi[k])
        if v == 0:
            return max(Hv)
        p = parent[v]
        if acc[p] is None:
            h0 = [NEG] * (k + 1); l0 = [POS] * (k + 1)
            h0[k] = l0[k] = 0
            acc[p] = (h0, l0)
        chH, chL = to_distance(Hv, Lv, k)
        acc[p] = merge(acc[p][0], acc[p][1], chH, chL, k)

print("khung DP đã định nghĩa")

khung DP đã định nghĩa


### 7.2. Cách 2 — DP với gộp `O(k²)` (duyệt mọi cặp)

**Giải thích.** Với mỗi cặp `(a, b)` (khoảng cách hiện tại `a`, khoảng cách của con `b`), nếu `a + b ≥ k` thì cặp hợp lệ, trạng thái mới `t = min(a, b)`, cập nhật `nh[t] = max(nh[t], hi[a] + chH[b])` và `nl[t] = min(nl[t], lo[a] + chL[b])`.

In [10]:
def merge_naive(hi, lo, chH, chL, k):
    nh = [NEG] * (k + 1)
    nl = [POS] * (k + 1)
    for a in range(1, k + 1):
        for b in range(1, k + 1):
            if a + b >= k:
                t = min(a, b)
                nh[t] = max(nh[t], hi[a] + chH[b])
                nl[t] = min(nl[t], lo[a] + chL[b])
    return nh, nl

def dp_naive(edges, nums, k):
    return tree_dp(len(nums), edges, nums, k, merge_naive)

for (edges, nums, k), exp in EXAMPLES:
    got = dp_naive(edges, nums, k)
    print(got, "(kỳ vọng", exp, ")")
    assert got == exp

23 (kỳ vọng 23 )
25 (kỳ vọng 25 )
12 (kỳ vọng 12 )
10 (kỳ vọng 10 )


**Độ phức tạp:** `O(n · k²)` thời gian (mỗi cạnh `k²` cặp, hằng số 2 cho max/min), `O(k · width)` bộ nhớ. Với `n = 5·10^4`, `k = 50`: cỡ `2.5·10^8` phép thử → quá chậm. Nhưng đủ để kiểm chứng.

### 7.3. Cách 3 — DP với gộp `O(k)` bằng max/min hậu tố

**Giải thích.** Với `t` cố định, các cặp `(cur, D)` cho `cur' = t` là:

- **(A)** `cur = t` và `D ≥ need`, với `need = max(t, k − t)`: lấy `hi[t] + (max chH[D] với D ≥ need)`.
- **(B)** `D = t` và `cur ≥ need`: lấy `chH[t] + (max hi[a] với a ≥ need)`.

(Cặp `cur = D = t` được tính hai lần, không sao vì ta lấy max/min.) Điều kiện `cur + D ≥ k`: khi `cur = t`, cần `D ≥ k − t`; kết hợp với `D ≥ t` để min đúng là `t`, ra `D ≥ max(t, k−t)`.

Chuẩn bị 4 mảng hậu tố: `sh` (max của `chH`), `sl` (min của `chL`), `ch2` (max của `hi`), `cl2` (min của `lo`). Mỗi mảng `O(k)`, rồi mỗi `t` tốn `O(1)`.

In [11]:
def merge_fast(hi, lo, chH, chL, k):
    sh = [NEG] * (k + 2); sl = [POS] * (k + 2)       # hậu tố trên chH / chL
    ph = [NEG] * (k + 2); pl = [POS] * (k + 2)       # hậu tố trên hi / lo
    for D in range(k, 0, -1):
        sh[D] = max(sh[D + 1], chH[D]); sl[D] = min(sl[D + 1], chL[D])
        ph[D] = max(ph[D + 1], hi[D]);  pl[D] = min(pl[D + 1], lo[D])
    nh = [NEG] * (k + 1)
    nl = [POS] * (k + 1)
    for t in range(1, k + 1):
        need = max(t, k - t)
        nh[t] = max(hi[t] + sh[need], chH[t] + ph[need])
        nl[t] = min(lo[t] + sl[need], chL[t] + pl[need])
    return nh, nl

def dp_fast(edges, nums, k):
    return tree_dp(len(nums), edges, nums, k, merge_fast)

for (edges, nums, k), exp in EXAMPLES:
    got = dp_fast(edges, nums, k)
    print(got, "(kỳ vọng", exp, ")")
    assert got == exp

23 (kỳ vọng 23 )
25 (kỳ vọng 25 )
12 (kỳ vọng 12 )
10 (kỳ vọng 10 )


**Độ phức tạp:** `O(n · k)` thời gian, `O(k · width)` bộ nhớ (mỗi nút giữ hai mảng độ dài `k + 1` cho đến khi gộp vào cha). Với `n = 5·10^4`, `k = 50`: khoảng vài triệu phép → chạy trong vài giây.

Đây cũng là lời giải ta dùng để nộp.

## 8. TRACE từng bước

Trace Ví dụ 4: `edges = [[0,1],[0,2]]`, `nums = [1,−5,−6]`, `k = 3`. Các mảng dưới đây được in từ chỉ số `0` đến `k`: `H[v][m]` (max tổng cây con) với `m = 0` nghĩa "chọn `v`"; `-` là không khả thi.

Kỳ vọng tay:

| Nút | Không chọn (`m = 3`) | Chọn (`m = 0`) |
|---|---|---|
| 1 (lá, −5) | `H = −5` | `H = 5` |
| 2 (lá, −6) | `H = −6` | `H = 6` |
| 0 | gộp 2 lá: `cur = 1` có `H = 1 + 6 + (−5) = 2` (đảo lá 2); `cur = 3` có `H = −10` | `H[0] = −(1 + (−11)) = 10` |

Đảo cả hai lá (`cur = 1 + 1 = 2 < 3`) bị loại ở bước gộp.

In [12]:
def trace_dp(edges, nums, k):
    n = len(nums)
    parent, order = rooted_tree(n, edges)
    fmt = lambda arr: "[" + ", ".join("-" if x in (NEG, POS) else str(x) for x in arr) + "]"
    acc = [None] * n
    for v in reversed(order):
        if acc[v] is None:
            hi = [NEG] * (k + 1); lo = [POS] * (k + 1); hi[k] = lo[k] = 0
        else:
            hi, lo = acc[v]; acc[v] = None
        Hv = [NEG] * (k + 1); Lv = [POS] * (k + 1)
        for t in range(1, k + 1):
            Hv[t] = nums[v] + hi[t]; Lv[t] = nums[v] + lo[t]
        Hv[0] = -(nums[v] + lo[k]); Lv[0] = -(nums[v] + hi[k])
        print(f"nút {v} (giá trị {nums[v]}): tổng các con hi={fmt(hi[1:])} lo={fmt(lo[1:])}  (chỉ số 1..k)")
        print(f"        H[{v}] = {fmt(Hv)}   L[{v}] = {fmt(Lv)}")
        if v == 0:
            print("đáp án = max(H[0]) =", max(Hv))
            return max(Hv)
        p = parent[v]
        if acc[p] is None:
            h0 = [NEG] * (k + 1); l0 = [POS] * (k + 1); h0[k] = l0[k] = 0
            acc[p] = (h0, l0)
        chH, chL = to_distance(Hv, Lv, k)
        print(f"        chuyển sang khoảng cách từ cha {p}: chH={fmt(chH[1:])}")
        acc[p] = merge_fast(acc[p][0], acc[p][1], chH, chL, k)
        print(f"        sau khi gộp vào cha {p}: hi={fmt(acc[p][0][1:])}")

assert trace_dp([[0,1],[0,2]], [1,-5,-6], 3) == 10

nút 2 (giá trị -6): tổng các con hi=[-, -, 0] lo=[-, -, 0]  (chỉ số 1..k)
        H[2] = [6, -, -, -6]   L[2] = [6, -, -, -6]
        chuyển sang khoảng cách từ cha 0: chH=[6, -, -6]
        sau khi gộp vào cha 0: hi=[6, -, -6]
nút 1 (giá trị -5): tổng các con hi=[-, -, 0] lo=[-, -, 0]  (chỉ số 1..k)
        H[1] = [5, -, -, -5]   L[1] = [5, -, -, -5]
        chuyển sang khoảng cách từ cha 0: chH=[5, -, -5]
        sau khi gộp vào cha 0: hi=[1, -, -11]
nút 0 (giá trị 1): tổng các con hi=[1, -, -11] lo=[-1, -, -11]  (chỉ số 1..k)
        H[0] = [10, 2, -, -10]   L[0] = [10, 0, -, -10]
đáp án = max(H[0]) = 10


## 9. Code nộp LeetCode

Gộp thành một `class Solution` độc lập (không phụ thuộc hàm đã định nghĩa ở trên). Tên phương thức lấy theo suy luận — **cần đối chiếu với skeleton trên LeetCode** (bài Premium).

In [13]:
from typing import List

class Solution:
    def subtreeInversionSum(self, edges: List[List[int]], nums: List[int], k: int) -> int:
        n = len(nums)
        NEG, POS = float("-inf"), float("inf")

        adj = [[] for _ in range(n)]
        for u, v in edges:
            adj[u].append(v)
            adj[v].append(u)
        parent = [-1] * n
        order = [0]
        seen = [False] * n
        seen[0] = True
        for u in order:
            for w in adj[u]:
                if not seen[w]:
                    seen[w] = True
                    parent[w] = u
                    order.append(w)

        def fresh():
            hi = [NEG] * (k + 1)
            lo = [POS] * (k + 1)
            hi[k] = lo[k] = 0
            return hi, lo

        acc = [None] * n
        for v in reversed(order):
            hi, lo = acc[v] if acc[v] is not None else fresh()
            acc[v] = None
            Hv = [NEG] * (k + 1)
            Lv = [POS] * (k + 1)
            x = nums[v]
            for t in range(1, k + 1):
                Hv[t] = x + hi[t]
                Lv[t] = x + lo[t]
            Hv[0] = -(x + lo[k])
            Lv[0] = -(x + hi[k])
            if v == 0:
                return int(max(Hv))
            # chuyển sang "khoảng cách từ cha"
            chH = [NEG] * (k + 1)
            chL = [POS] * (k + 1)
            for D in range(1, k):
                chH[D] = Hv[D - 1]
                chL[D] = Lv[D - 1]
            chH[k] = max(Hv[k - 1], Hv[k])
            chL[k] = min(Lv[k - 1], Lv[k])
            # gộp vào bộ tích lũy của cha bằng max/min hậu tố
            p = parent[v]
            ah, al = acc[p] if acc[p] is not None else fresh()
            sh = [NEG] * (k + 2); sl = [POS] * (k + 2)
            ph = [NEG] * (k + 2); pl = [POS] * (k + 2)
            for D in range(k, 0, -1):
                sh[D] = max(sh[D + 1], chH[D]); sl[D] = min(sl[D + 1], chL[D])
                ph[D] = max(ph[D + 1], ah[D]);  pl[D] = min(pl[D + 1], al[D])
            nh = [NEG] * (k + 1)
            nl = [POS] * (k + 1)
            for t in range(1, k + 1):
                need = t if t > k - t else k - t
                nh[t] = max(ah[t] + sh[need], chH[t] + ph[need])
                nl[t] = min(al[t] + sl[need], chL[t] + pl[need])
            acc[p] = (nh, nl)
        return 0

sol = Solution()
for (edges, nums, k), exp in EXAMPLES:
    got = sol.subtreeInversionSum(edges, nums, k)
    print(got)
    assert got == exp

23
25
12
10


## 10. Vì sao thuật toán đúng

**Bất biến.** Sau khi xử lý xong nút `v`, `H[v][m]` (tương ứng `L[v][m]`) là max (min) của tổng giá trị cây con của `v` trên mọi cách chọn tập nút **trong cây con** thỏa mọi ràng buộc khoảng cách giữa các nút trong cây con, mà khoảng cách từ `v` tới nút chọn gần nhất bằng `m` (chặn ở `k`).

**Chứng minh quy nạp theo thứ tự gộp.**

1. **Hai nút chọn trong cùng một cây con của con `c`:** đã được kiểm tra khi xử lý `c` (giả thiết quy nạp).
2. **Hai nút chọn thuộc hai cây con anh em `c₁, c₂`:** đường đi qua `v`, khoảng cách bằng `d₁ + d₂` với `dᵢ ≥` khoảng cách từ `v` tới nút chọn gần nhất trong cây con `cᵢ`. Điều kiện `cur + D ≥ k` được kiểm tra bằng đúng hai nút **gần nhất**; nếu hai nút gần nhất đã thỏa thì mọi cặp khác (xa hơn) cũng thỏa; nếu không thỏa thì cặp gần nhất là bằng chứng vi phạm.
3. **Nút `v` được chọn với nút trong cây con:** khoảng cách từ `v` tới nút gần nhất phải `≥ k`, đó là điều kiện `t = k` (đã chặn: mọi giá trị `≥ k` gộp lại).
4. **Nút chọn ngoài cây con của `v`:** cách `v` ≥ 1 và đường đi tới nút trong cây con **đi qua `v`**; khoảng cách = `dist(ngoài, v) + dist(v, trong)`. Trạng thái `m` (khoảng cách tới nút gần nhất) đủ để kiểm tra điều kiện ở cấp trên, đúng như bước 2/3 tại tổ tiên chung.

**Dấu.** Nếu `v` được chọn, mọi nút của cây con bị đổi dấu thêm một lần, kể cả các nút đã có dấu đổi từ trong cây con. Tổng của cây con khi đó là `−(tổng khi chưa tính đảo tại v)`. Để max hóa `−x` ta cần min của `x`; vì vậy phải duy trì cả max và min. Hơn nữa tập chọn bên trong cây con của các con độc lập nhau (cùng nằm dưới `v`, mỗi tập chỉ phụ thuộc cây con riêng) nên max (min) của tổng là tổng các max (min) — **bước gộp chỉ ràng buộc qua khoảng cách `m`**, vì thế max/min trên mỗi `m` là đủ.

Bài test vét cạn ở mục 12 xác nhận điều này trên nhiều cây ngẫu nhiên.

## 11. Phân tích Time/Space

Đặt `n` là số nút, `k ≤ 50`.

| Phần | Thời gian | Không gian |
|---|---|---|
| Dựng cây (BFS) | `O(n)` | `O(n)` |
| Mỗi nút: tạo `H, L` | `O(k)` | `O(k)` |
| Mỗi cạnh: `to_distance` + gộp bằng hậu tố | `O(k)` | `O(k)` |

**Tổng:** `O(n · k)` thời gian (≈ `5·10^4 · 50 · hằng số ≈ vài chục triệu` phép đơn giản), không gian `O(n + k · W)` với `W` là số nút có bộ tích lũy đang mở (≤ độ rộng lớn nhất của một tầng BFS). Vì ta giải phóng mảng của nút ngay sau khi gộp vào cha, bộ nhớ không phình ra `O(n · k)` trong trường hợp xấu nhất khác thường.

So với cách 2: `O(n · k²)`.

## 12. Bộ test

Ví dụ đề, edge case (`k = 1`, `k` lớn hơn đường kính, cây đường thẳng, cây sao), phản ví dụ cho các cách làm sai, và đối chiếu ngẫu nhiên có seed với brute force.

In [14]:
def gen_tree(n, shape, rng):
    # trả về danh sách cạnh với nút gốc 0, nhãn đã xáo trộn nhẹ (cạnh vô hướng, thứ tự ngẫu nhiên)
    if shape == "path":
        raw = [(i - 1, i) for i in range(1, n)]
    elif shape == "star":
        raw = [(0, i) for i in range(1, n)]
    elif shape == "binary":
        raw = [((i - 1) // 2, i) for i in range(1, n)]
    else:
        raw = [(rng.randrange(i), i) for i in range(1, n)]
    perm = list(range(n))
    rest = perm[1:]
    rng.shuffle(rest)
    perm = [0] + rest                 # giữ gốc là 0 nhưng xáo nhãn còn lại
    edges = []
    for a, b in raw:
        u, v = perm[a], perm[b]
        edges.append([u, v] if rng.random() < 0.5 else [v, u])
    rng.shuffle(edges)
    return edges

# 12.1 Ví dụ đề + edge case
sol = Solution()
for (edges, nums, k), exp in EXAMPLES:
    assert sol.subtreeInversionSum(edges, nums, k) == exp
assert sol.subtreeInversionSum([[0, 1]], [5, 7], 1) == 12                 # n=2, không cần đảo
assert sol.subtreeInversionSum([[0, 1]], [-5, -7], 1) == 12               # đảo nút 0
assert sol.subtreeInversionSum([[0, 1], [1, 2], [2, 3]], [-1, -1, -1, -1], 50) == 4   # k lớn: chọn nút 0 đảo hết
assert sol.subtreeInversionSum([[0, 1], [0, 2], [0, 3]], [0, -1, -1, -1], 2) == 3     # sao: đảo cả ba lá (cách nhau 2)
print("ví dụ + edge case OK")

ví dụ + edge case OK


In [15]:
# 12.2 DP đối chiếu vét cạn trên cây nhỏ (n <= 11)
rng = random.Random(3949)
cnt = 0
for _ in range(250):
    n = rng.randint(2, 11)
    k = rng.randint(1, 7)
    edges = gen_tree(n, rng.choice(["random", "path", "star", "binary"]), rng)
    nums = [rng.randint(-9, 9) for _ in range(n)]
    exp = brute_force(edges, nums, k)
    assert dp_naive(edges, nums, k) == exp, (edges, nums, k)
    assert dp_fast(edges, nums, k) == exp, (edges, nums, k)
    assert sol.subtreeInversionSum(edges, nums, k) == exp, (edges, nums, k)
    cnt += 1
print(cnt, "test nhỏ: 3 cách khớp nhau")

250 test nhỏ: 3 cách khớp nhau


In [16]:
# 12.3 Hai cách DP đối chiếu nhau trên cây vừa (n <= 80, k <= 20, giá trị lớn)
rng = random.Random(2026)
for _ in range(120):
    n = rng.randint(2, 80)
    k = rng.randint(1, 20)
    edges = gen_tree(n, rng.choice(["random", "path", "star", "binary"]), rng)
    nums = [rng.randint(-40000, 40000) for _ in range(n)]
    a = dp_naive(edges, nums, k)
    b = dp_fast(edges, nums, k)
    assert a == b == sol.subtreeInversionSum(edges, nums, k), (n, k)
print("120 test vừa: DP O(k^2) = DP O(k) = Solution")

# 12.4 Thời gian với dữ liệu lớn nhất
rng = random.Random(1)
n, k = 50000, 50
for shape in ("random", "path", "star", "binary"):
    edges = gen_tree(n, shape, rng)
    nums = [rng.randint(-40000, 40000) for _ in range(n)]
    t0 = time.time()
    r = sol.subtreeInversionSum(edges, nums, k)
    print(f"{shape:7s} n=5e4 k=50 -> {r}  ({time.time() - t0:.2f}s)")

120 test vừa: DP O(k^2) = DP O(k) = Solution


random  n=5e4 k=50 -> 4478990  (1.46s)


path    n=5e4 k=50 -> 209834691  (1.35s)


star    n=5e4 k=50 -> 3212519  (1.41s)


binary  n=5e4 k=50 -> 10862069  (1.45s)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra khoảng cách tổ tiên – hậu duệ, bỏ qua anh em/họ hàng

**Code sai:** gộp con mà không kiểm tra `cur + D ≥ k` (chỉ kiểm tra khi chọn `v`).
**Vì sao sai:** đề yêu cầu **mọi cặp** nút được đảo cách nhau ≥ k, kể cả hai nút ở hai nhánh khác nhau.
**Code đúng:** điều kiện `a + b >= k` trong phép gộp.

In [17]:
def merge_no_pair_check(hi, lo, chH, chL, k):
    nh = [NEG] * (k + 1); nl = [POS] * (k + 1)
    for a in range(1, k + 1):
        for b in range(1, k + 1):
            t = min(a, b)                          # SAI: thiếu điều kiện a + b >= k
            nh[t] = max(nh[t], hi[a] + chH[b])
            nl[t] = min(nl[t], lo[a] + chL[b])
    return nh, nl

edges, nums, k = [[0, 1], [0, 2]], [1, -5, -6], 3
print("sai :", tree_dp(3, edges, nums, k, merge_no_pair_check), " đúng:", dp_fast(edges, nums, k))

sai : 12  đúng: 10


### Lỗi 2 — Dùng `>` thay vì `>=` (khoảng cách phải **ít nhất** `k`)

**Code sai:** `a + b > k`.
**Vì sao sai:** hai nút cách đúng `k` là hợp lệ (Ví dụ 3: `k = 2`, hai lá cách 2).
**Code đúng:** `a + b >= k`.

In [18]:
def merge_strict(hi, lo, chH, chL, k):
    nh = [NEG] * (k + 1); nl = [POS] * (k + 1)
    for a in range(1, k + 1):
        for b in range(1, k + 1):
            if a + b > k:                          # SAI: phải là >=
                t = min(a, b)
                nh[t] = max(nh[t], hi[a] + chH[b])
                nl[t] = min(nl[t], lo[a] + chL[b])
    return nh, nl

edges, nums, k = [[0, 1], [0, 2]], [1, -5, -6], 2
print("sai :", tree_dp(3, edges, nums, k, merge_strict), " đúng:", dp_fast(edges, nums, k))

sai : 10  đúng: 12


### Lỗi 3 — Chỉ lưu max, quên min

**Code sai:** khi chọn `v` dùng `H[v][0] = −(nums[v] + hi[k])` (lấy max rồi đảo dấu).
**Vì sao sai:** `max(−x) = −min(x)`; max của tổng con không cho ta max sau khi đảo.
**Code đúng:** `H[v][0] = −(nums[v] + lo[k])`.

In [19]:
def tree_dp_only_max(n, edges, nums, k):
    parent, order = rooted_tree(n, edges)
    acc = [None] * n
    for v in reversed(order):
        if acc[v] is None:
            hi = [NEG] * (k + 1); lo = [POS] * (k + 1); hi[k] = lo[k] = 0
        else:
            hi, lo = acc[v]; acc[v] = None
        Hv = [NEG] * (k + 1); Lv = [POS] * (k + 1)
        for t in range(1, k + 1):
            Hv[t] = nums[v] + hi[t]; Lv[t] = nums[v] + lo[t]
        Hv[0] = -(nums[v] + hi[k])                 # SAI: dùng hi thay cho lo
        Lv[0] = -(nums[v] + lo[k])
        if v == 0:
            return max(Hv)
        p = parent[v]
        if acc[p] is None:
            h0 = [NEG] * (k + 1); l0 = [POS] * (k + 1); h0[k] = l0[k] = 0
            acc[p] = (h0, l0)
        chH, chL = to_distance(Hv, Lv, k)
        acc[p] = merge_fast(acc[p][0], acc[p][1], chH, chL, k)

rng = random.Random(5)
found = None
for _ in range(2000):
    n = rng.randint(2, 7); k = rng.randint(1, 3)
    edges = gen_tree(n, "random", rng)
    nums = [rng.randint(-5, 5) for _ in range(n)]
    if tree_dp_only_max(n, edges, nums, k) != brute_force(edges, nums, k):
        found = (edges, nums, k); break
edges, nums, k = found
print("phản ví dụ:", found)
print("sai :", tree_dp_only_max(len(nums), edges, nums, k), " đúng:", brute_force(edges, nums, k))

phản ví dụ: ([[1, 0], [2, 0]], [-3, -1, 0], 1)
sai : 2  đúng: 4


### Lỗi 4 — Đệ quy DFS trên cây dài 5·10^4 nút

Python mặc định giới hạn độ sâu đệ quy khoảng 1000. Một cây dạng đường thẳng có độ sâu `5·10^4` sẽ gây `RecursionError`. Hãy duyệt theo thứ tự BFS ngược (như trong notebook) hoặc dùng stack tường minh.

In [20]:
import sys
def depth_recursive(d):
    return 0 if d == 0 else 1 + depth_recursive(d - 1)

print("giới hạn đệ quy:", sys.getrecursionlimit())
try:
    depth_recursive(10**5)
except RecursionError:
    print("RecursionError với độ sâu 1e5 -> không dùng đệ quy cho cây dạng đường thẳng")

giới hạn đệ quy: 1000
RecursionError với độ sâu 1e5 -> không dùng đệ quy cho cây dạng đường thẳng


### Lỗi 5 — Không chặn khoảng cách ở `k`

Nếu lưu `m` tới `n` thì bảng DP có `n²` ô — hết bộ nhớ/thời gian. Mọi khoảng cách `≥ k` tương đương nhau, nên chặn ở `k`.

## 14. Mẹo & nhận dạng pattern

- **"Chọn tập nút sao cho hai nút bất kỳ cách nhau ≥ k"** → DP cây với trạng thái "khoảng cách tới nút chọn gần nhất trong cây con", chặn ở `k`. Mẫu này cũng dùng cho bài đặt camera, bài "dominating set" nhỏ.
- Phép biến đổi **nhân −1 cả cây con** → dấu là parity của số tổ tiên được chọn; trong DP cây con ta lưu max **và** min vì phép nhân `−1` đổi chỗ chúng.
- Phép gộp "hai nhánh phải cách nhau ≥ k" có dạng `a + b ≥ k`, `t = min(a, b)`. Dạng này tối ưu bằng **max/min hậu tố** theo ngưỡng `max(t, k − t)`.
- Cây cho dạng cạnh vô hướng: BFS dựng cha rồi duyệt ngược; tránh đệ quy.
- Giải phóng mảng DP của con ngay sau khi gộp vào cha để tiết kiệm bộ nhớ.

## 15. Biến thể

1. **Phiên bản chỉ ràng buộc cặp tổ tiên – hậu duệ** (theo hiểu biết của tôi là cách đề bài phiên bản I phát biểu): không cần kiểm tra `a + b ≥ k` khi gộp anh em, chỉ cần trạng thái "khoảng cách tới tổ tiên được chọn gần nhất" truyền từ trên xuống.
2. **Số phép đảo tối đa `m`:** thêm một chiều trạng thái `j` = số nút đã chọn → `O(n · k · m)`.
3. **Mỗi nút đảo tốn chi phí `c[v]`:** trừ chi phí khi chọn `v` ở bước 6.3.
4. **Cây có trọng số cạnh:** khoảng cách tính theo trọng số → khó hơn vì không thể chặn bằng chỉ số nguyên nhỏ (cần nén hoặc dùng cấu trúc khác).

In [21]:
# Biến thể 1 (minh họa): chỉ cặp tổ tiên - hậu duệ bị ràng buộc -> bỏ điều kiện a+b>=k khi gộp anh em
# (đây KHÔNG phải đề gốc: ví dụ 4 khi đó cho 12 thay vì 10)
edges, nums, k = [[0, 1], [0, 2]], [1, -5, -6], 3
print("ràng buộc cả cặp anh em:", dp_fast(edges, nums, k))
print("chỉ tổ tiên - hậu duệ  :", tree_dp(3, edges, nums, k, merge_no_pair_check))

ràng buộc cả cặp anh em: 10
chỉ tổ tiên - hậu duệ  : 12


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn tập con | thử `2^n` tập, kiểm tra mọi cặp | `O(2^n · n²)` | `O(n²)` | kiểm chứng, `n ≤ 12` |
| 2. DP gộp `O(k²)` | trạng thái (nút, khoảng cách gần nhất, max/min); duyệt mọi cặp `(a, b)` | `O(n · k²)` | `O(k · W)` | dễ viết, `n·k²` nhỏ |
| 3. DP gộp `O(k)` | như trên nhưng dùng max/min hậu tố | `O(n · k)` | `O(k · W)` | **lời giải nộp** |

`W` = số nút có bộ tích lũy mở (≤ độ rộng tầng BFS lớn nhất).

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `final_values(3, [-1, 0, 0], [1, -5, -6], [0])` in ra gì? (nút gốc bị đảo.)

**Bài 2 (tính tay).** Cây đường thẳng `0 – 1 – 2`, `nums = [3, -4, -4]`, `k = 1`. Đáp án là bao nhiêu và chọn nút nào?

**Bài 3 (điền TODO).** Hoàn thiện `my_finish_node(x, hi, lo, k)` trả về cặp `(Hv, Lv)` theo mục 6.3.

**Bài 4 (tự viết lại).** Viết `my_subtree_inversion(edges, nums, k)` bằng DP gộp `O(k²)`, không nhìn code.

**Bài 5 (biến thể).** Viết `with_budget(edges, nums, k, m)`: tối đa `m` nút được đảo (vét cạn cho `n ≤ 10` cũng được).

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_finish_node(x, hi, lo, k):
    # TODO 1: Hv, Lv = mảng độ dài k+1, khởi tạo -inf / +inf
    # TODO 2: với t in 1..k: Hv[t] = x + hi[t]; Lv[t] = x + lo[t]      (không chọn nút)
    # TODO 3: Hv[0] = -(x + lo[k]); Lv[0] = -(x + hi[k])                  (chọn nút)
    pass

# Bài 4 — TODO
def my_subtree_inversion(edges, nums, k):
    # TODO: dựng cha bằng BFS, duyệt ngược, gộp con vào cha bằng hai vòng for lồng (a, b), điều kiện a + b >= k
    pass

# Bài 5 — TODO
def with_budget(edges, nums, k, m):
    # TODO: vét cạn tập con có không quá m phần tử, hợp lệ theo khoảng cách
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: nút gốc được chọn → mọi nút bị nhân `−1` một lần.
- Mức 2: `[1, −5, −6]` đảo dấu từng phần tử.
- Mức 3: kết quả `[-1, 5, 6]`.

**Bài 2.**
- Mức 1: `k = 1` nên mọi cặp (khác nút) đều thỏa khoảng cách.
- Mức 2: thử `{1}`: `3 + 4 − 4 = 3`; `{1, 2}`: nút 2 bị đảo hai lần (qua 1 và qua 2)...
- Mức 3: `{1}`: nút 1 và nút 2 đều đảo → `3 + 4 + 4 = 11`; `{2}`: `3 − 4 + 4 = 3`; `{1,2}`: nút 2 đảo hai lần → `3 + 4 − 4 = 3`. Đáp án `11`, chọn nút 1.

**Bài 3.**
- Mức 1: hai vòng đơn giản; phần "chọn nút" chỉ có một ô `[0]`.
- Mức 2: chọn nút cần mọi nút chọn trong con cách ≥ k → dùng ô `hi[k]`, `lo[k]`.
- Mức 3: `Hv[0] = -(x + lo[k])`.

**Bài 4.**
- Mức 1: xem lại `tree_dp` và `merge_naive`.
- Mức 2: khởi tạo `hi[k] = lo[k] = 0`.
- Mức 3: kết quả là `max(Hv)` tại nút `0`.

**Bài 5.**
- Mức 1: duyệt mọi tập con.
- Mức 2: bỏ tập có hơn `m` phần tử.
- Mức 3: dùng `final_values` và `all_distances`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
assert final_values(3, [-1, 0, 0], [1, -5, -6], [0]) == [-1, 5, 6]
# Bài 2
assert brute_force([[0, 1], [1, 2]], [3, -4, -4], 1) == 11 == Solution().subtreeInversionSum([[0, 1], [1, 2]], [3, -4, -4], 1)

# Bài 3
def my_finish_node_ans(x, hi, lo, k):
    Hv = [NEG] * (k + 1); Lv = [POS] * (k + 1)
    for t in range(1, k + 1):
        Hv[t] = x + hi[t]; Lv[t] = x + lo[t]
    Hv[0] = -(x + lo[k]); Lv[0] = -(x + hi[k])
    return Hv, Lv

hi = [NEG, 4, NEG, 2]; lo = [POS, -1, POS, -3]
Hv, Lv = my_finish_node_ans(10, hi, lo, 3)
assert Hv == [-(10 - 3), 14, NEG, 12] and Lv == [-(10 + 2), 9, POS, 7]
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
def my_subtree_inversion_ans(edges, nums, k):
    n = len(nums)
    parent, order = rooted_tree(n, edges)
    def fresh():
        h = [NEG] * (k + 1); l = [POS] * (k + 1); h[k] = l[k] = 0
        return h, l
    acc = [None] * n
    for v in reversed(order):
        hi, lo = acc[v] if acc[v] else fresh()
        Hv, Lv = my_finish_node_ans(nums[v], hi, lo, k)
        if v == 0:
            return max(Hv)
        chH, chL = to_distance(Hv, Lv, k)
        p = parent[v]
        ah, al = acc[p] if acc[p] else fresh()
        nh = [NEG] * (k + 1); nl = [POS] * (k + 1)
        for a in range(1, k + 1):
            for b in range(1, k + 1):
                if a + b >= k:
                    t = min(a, b)
                    nh[t] = max(nh[t], ah[a] + chH[b])
                    nl[t] = min(nl[t], al[a] + chL[b])
        acc[p] = (nh, nl)

rng = random.Random(41)
for _ in range(80):
    n = rng.randint(2, 10); k = rng.randint(1, 6)
    edges = gen_tree(n, "random", rng)
    nums = [rng.randint(-9, 9) for _ in range(n)]
    assert my_subtree_inversion_ans(edges, nums, k) == brute_force(edges, nums, k)
print("Bài 4 OK")

Bài 4 OK


In [25]:
# Bài 5
def with_budget_ans(edges, nums, k, m):
    n = len(nums)
    parent, order = rooted_tree(n, edges)
    dist = all_distances(n, edges)
    best = None
    for r in range(0, min(m, n) + 1):
        for S in combinations(range(n), r):
            if any(dist[a][b] < k for a, b in combinations(S, 2)):
                continue
            tot = sum(final_values(n, parent, nums, S))
            best = tot if best is None else max(best, tot)
    return best

edges, nums, k = [[0, 1], [0, 2]], [1, -5, -6], 2
assert with_budget_ans(edges, nums, k, 2) == 12
assert with_budget_ans(edges, nums, k, 1) == 10       # chỉ đảo được 1 nút: đảo nút 0 cho -1+5+6 = 10
assert with_budget_ans(edges, nums, k, 0) == -10
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (DP cây: trạng thái = khoảng cách gần nhất, chặn ở k; max + min):**

```python
parent, order = BFS từ 0
acc = [None] * n
for v in reversed(order):
    hi, lo = acc[v] hoặc (hi[k]=lo[k]=0, còn lại -inf/+inf)
    Hv[t] = nums[v] + hi[t], Lv[t] = nums[v] + lo[t]   # t = 1..k  (không chọn v)
    Hv[0] = -(nums[v] + lo[k]); Lv[0] = -(nums[v] + hi[k])   # chọn v
    if v == 0: return max(Hv)
    chuyển sang D = min(m+1, k); gộp vào acc[parent[v]]:
        nh[t] = max(hi[t] + sufmax_ch[max(t,k-t)],  ch[t] + sufmax_hi[max(t,k-t)])
        nl[t] = min(...)  tương tự
```

**Checklist trước khi submit:**

- [ ] Dựng cha bằng BFS (cạnh vô hướng), duyệt ngược, **không đệ quy**.
- [ ] `m = 0` là "chọn nút", `m = k` là "không có hoặc xa ≥ k".
- [ ] Gộp con: `D = min(m_c + 1, k)`, điều kiện `cur + D ≥ k` (dấu `≥`).
- [ ] Chọn nút chỉ khi `cur = k`; khi đó dùng **min** của tổng con để có max sau khi đảo.
- [ ] Lưu cả max và min.
- [ ] Đáp án = `max(H[0])` (cho phép tập rỗng, nằm trong trạng thái `m = k`).
- [ ] Đã đối chiếu brute force `n ≤ 11` trên nhiều cây (path, star, binary, random).
- [ ] Đã đối chiếu chữ ký phương thức Premium trên LeetCode.